# GANIA 2.0 · Entrenamiento FastGAN en Google Colab (GPU)

Notebook para entrenar **tu FastGAN de GANIA 2.0** (con DiffAugment + EMA) en la GPU de Colab y traer el resultado de vuelta a la app de escritorio.

Usa directamente tu código: `modelos/fastgan.py`, `utils/cargador_datos.py`, `utils/metricas_ia.py` y `config.py` (no necesita PyQt5).

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.

| Paso | Qué hace |
|---|---|
| 1 | Verifica la GPU |
| 2 | Parámetros (clase, épocas, batch...) |
| 3 | Monta Google Drive (para no perder checkpoints si Colab se desconecta) |
| 4 | Obtiene tu código (GitHub o ZIP) |
| 5 | Obtiene el dataset (URL de Oxford 17, Drive o ZIP) |
| 6 | Entrena con vista previa y métricas en vivo (con reanudación) |
| 7 | Genera imágenes sintéticas y descarga ZIP + checkpoint |

## 1. Verificar GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print("PyTorch:", torch.__version__, "| CUDA disponible:", torch.cuda.is_available())
assert torch.cuda.is_available(), "⚠ No hay GPU: ve a Entorno de ejecución → Cambiar tipo de entorno → GPU"

## 2. Parámetros
Modifica estos valores. Con `ITERACIONES_POR_EPOCA = 50`, **500 épocas = 25.000 iteraciones** (igual que en la app).

In [ ]:
# ---------- Código ----------
ORIGEN_CODIGO = "github"        # "github" | "zip_drive" | "zip_subir"
GITHUB_URL    = "https://github.com/LeonidCJ/GANIA-2.0.git"
GITHUB_RAMA   = "main"
ZIP_CODIGO_DRIVE = "/content/drive/MyDrive/GANIA/gania_codigo.zip"   # si ORIGEN_CODIGO = "zip_drive"

# ---------- Dataset ----------
ORIGEN_DATASET = "url"          # "url" | "carpeta_drive" | "zip_drive" | "zip_subir"
URL_DATASET    = "https://www.robots.ox.ac.uk/~vgg/data/flowers/17/17flowers.tgz"
RUTA_DATASET_DRIVE = "/content/drive/MyDrive/GANIA/Oxford Flowers 17"  # carpeta o .zip/.tgz en Drive
CLASE = "snowdrop"              # clase a entrenar (se listan en el paso 5)

# ---------- Entrenamiento ----------
EPOCAS = 500
RESOLUCION = 128                # 64 | 128 | 256 (en GPU de Colab 128 o 256 dan mucha mayor nitidez)
BATCH_SIZE = 32
LR = 0.0002
ITERACIONES_POR_EPOCA = 50
INTERVALO_VISTA_PREVIA = 5      # mostrar rejilla cada N épocas
INTERVALO_METRICAS = 10         # KID/FID/Diversidad cada N épocas
MUESTRAS_METRICAS = 256
INTERVALO_CHECKPOINT = 25
REANUDAR = True                 # continuar desde el último checkpoint de esta clase si existe

# ---------- Salida ----------
USAR_DRIVE = True               # guardar checkpoints/resultados en Drive (recomendado)
CANTIDAD_SINTETICAS = 500

## 3. Google Drive (opcional, recomendado)

In [ ]:
import os
necesita_drive = USAR_DRIVE or ORIGEN_CODIGO == "zip_drive" or ORIGEN_DATASET in ("carpeta_drive", "zip_drive")
if necesita_drive:
    from google.colab import drive
    drive.mount("/content/drive")

DIR_SALIDA = "/content/drive/MyDrive/GANIA/resultados_colab" if USAR_DRIVE else "/content/resultados_colab"
DIR_CHECKPOINTS = os.path.join(DIR_SALIDA, "checkpoints", CLASE)
os.makedirs(DIR_CHECKPOINTS, exist_ok=True)
print("Resultados en:", DIR_SALIDA)

## 4. Obtener tu código
- **github**: clona tu repo. ⚠ Asegúrate de haber hecho `git push` de los últimos cambios (DiffAugment, EMA, métricas...).
- **zip_drive / zip_subir**: un ZIP con `modelos/`, `utils/` y `config.py` (el proyecto incluye `colab/gania_codigo.zip`).

In [ ]:
import os, sys, shutil, zipfile
DIR_PROYECTO = "/content/GANIA"

if os.path.exists(DIR_PROYECTO):
    shutil.rmtree(DIR_PROYECTO)

if ORIGEN_CODIGO == "github":
    !git clone --depth 1 -b {GITHUB_RAMA} {GITHUB_URL} {DIR_PROYECTO}
else:
    if ORIGEN_CODIGO == "zip_subir":
        from google.colab import files
        subido = files.upload()
        ruta_zip = list(subido.keys())[0]
    else:
        ruta_zip = ZIP_CODIGO_DRIVE
    with zipfile.ZipFile(ruta_zip) as z:
        z.extractall(DIR_PROYECTO)
    # Si el ZIP trae una carpeta raíz, subir un nivel
    contenido = os.listdir(DIR_PROYECTO)
    if len(contenido) == 1 and os.path.isdir(os.path.join(DIR_PROYECTO, contenido[0])):
        DIR_PROYECTO = os.path.join(DIR_PROYECTO, contenido[0])

os.chdir(DIR_PROYECTO)
if DIR_PROYECTO not in sys.path:
    sys.path.insert(0, DIR_PROYECTO)

# Verificar que el código tiene las mejoras (DiffAugment + EMA)
import modelos.fastgan as fg
assert hasattr(fg, "DiffAugment") and hasattr(fg.EntrenadorFastGAN, "generar"), \
    "⚠ El código no tiene DiffAugment/EMA: haz git push de los últimos cambios o usa el ZIP."
print("Código cargado desde:", DIR_PROYECTO)

## 5. Dataset

In [ ]:
from utils.cargador_datos import (extraer_archivo_comprimido, descargar_extraer_url,
                                  preparar_y_analizar_dataset)

if ORIGEN_DATASET == "url":
    ruta_bruta = descargar_extraer_url(URL_DATASET, "/content/datos/url_extraido", callback_log=print)
elif ORIGEN_DATASET == "carpeta_drive":
    ruta_bruta = "/content/datos/copia_drive"
    if not os.path.exists(ruta_bruta):
        shutil.copytree(RUTA_DATASET_DRIVE, ruta_bruta)   # copia local: leer de Drive es lento
elif ORIGEN_DATASET == "zip_drive":
    ruta_bruta = extraer_archivo_comprimido(RUTA_DATASET_DRIVE, "/content/datos/zip_extraido")
else:  # zip_subir
    from google.colab import files
    subido = files.upload()
    ruta_bruta = extraer_archivo_comprimido(list(subido.keys())[0], "/content/datos/zip_extraido")

# Mismo módulo de preparación que la app (detecta Oxford 17, 102, carpetas por clase, etc.)
RUTA_DATASET, resumen = preparar_y_analizar_dataset(ruta_bruta)
clases = resumen.get("train") or resumen.get("total_por_clase") or {}
print("Formato:", resumen.get("formato"))
print("Ruta preparada:", RUTA_DATASET)
print("Clases:", {k: v for k, v in clases.items()})
assert CLASE in clases, f"⚠ La clase '{CLASE}' no existe. Elige una de: {list(clases)}"

## 6. Preparar datos, modelo y métricas
Se usa el mismo pipeline que la app (`obtener_transformaciones`, muestreo con reemplazo, DiffAugment, EMA),
pero las imágenes se **cachean en RAM ya redimensionadas** y se cargan con varios *workers*:
así la GPU no espera a que se decodifiquen los JPG en cada iteración.

In [ ]:
import glob, time
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader, RandomSampler

from config import CONFIG
from modelos.fastgan import EntrenadorFastGAN
from utils.cargador_datos import (DatasetClaseEspecifica, obtener_transformaciones,
                                  cargar_imagenes_reales_clase, _resolver_dir_clase)
from utils.metricas_ia import calcular_metricas_calidad, extraer_caracteristicas_reales
from utils.tensor_utils import tensor_to_numpy

dispositivo = torch.device("cuda")
torch.backends.cudnn.benchmark = True
TAM = RESOLUCION

class DatasetEnMemoria(Dataset):
    # Igual que DatasetClaseEspecifica pero con las imágenes precargadas (lado corto = TAM*1.125).
    def __init__(self, carpeta, tam):
        rutas = DatasetClaseEspecifica(carpeta).imgs
        tam_resize = int(round(tam * 1.125))
        self.imgs = []
        for r in rutas:
            img = Image.open(r).convert("RGB")
            img.thumbnail((10_000, 10_000))
            w, h = img.size
            esc = tam_resize / min(w, h)
            self.imgs.append(img.resize((max(tam_resize, round(w * esc)), max(tam_resize, round(h * esc))), Image.BICUBIC))
        self.transform = obtener_transformaciones(tam, normalizar_fastgan=True, para_entrenamiento=True)
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i): return self.transform(self.imgs[i]), 0

dir_clase = _resolver_dir_clase(RUTA_DATASET, CLASE)
dataset = DatasetEnMemoria(dir_clase, TAM)
sampler = RandomSampler(dataset, replacement=True, num_samples=ITERACIONES_POR_EPOCA * BATCH_SIZE)
cargador = DataLoader(dataset, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True,
                      num_workers=2, pin_memory=True, persistent_workers=True)
print(f"Clase '{CLASE}': {len(dataset)} imágenes | {ITERACIONES_POR_EPOCA} it/época × {EPOCAS} épocas = {ITERACIONES_POR_EPOCA*EPOCAS:,} iteraciones")

# Modelo (mismos hiperparámetros que la app)
cfg = CONFIG["FASTGAN"]
gan = EntrenadorFastGAN(dispositivo, lr=LR, tamano_img=TAM,
                        politica_aug=cfg.get("POLITICA_DIFFAUGMENT", "color,translation,cutout"),
                        ema_beta=cfg.get("EMA_BETA", 0.999))

# Reales para métricas (todas, sin aumentos) + features cacheadas
reales_metricas = cargar_imagenes_reales_clase(RUTA_DATASET, CLASE, TAM)
feats_reales = extraer_caracteristicas_reales(reales_metricas, "cuda")

# Historial (se guarda junto al checkpoint para poder reanudar las gráficas)
historial = {"epoca": [], "loss_G": [], "loss_D": [], "ep_met": [], "kid": [], "fid": [], "div": []}
epoca_inicio = 0

if REANUDAR:
    cks = sorted(glob.glob(os.path.join(DIR_CHECKPOINTS, "fastgan_epoca_*.pth")),
                 key=lambda p: int(p.split("_")[-1].split(".")[0]))
    if cks:
        epoca_inicio = gan.cargar_checkpoint(cks[-1])
        ruta_hist = os.path.join(DIR_CHECKPOINTS, "historial.pt")
        if os.path.exists(ruta_hist):
            historial = torch.load(ruta_hist)
        print(f"↻ Reanudando desde la época {epoca_inicio} ({os.path.basename(cks[-1])})")

## 7. Entrenar
Muestra la rejilla 3×3 (generador EMA), las curvas de pérdida y KID/Diversidad. Guarda checkpoints en Drive:
si Colab se desconecta, vuelve a ejecutar los pasos 1-6 con `REANUDAR = True` y continúa aquí.

**Cómo leerlo:** la pérdida de D no debería caer a ~0; KID×1000 debe bajar; Diversidad cerca de 100% (< 30% = *mode collapse*).

In [ ]:
import matplotlib.pyplot as plt
import torchvision.utils as vutils
from IPython.display import clear_output

def calcular_metricas():
    falsas = []
    while len(falsas) < MUESTRAS_METRICAS:
        n = min(128, MUESTRAS_METRICAS - len(falsas))
        z = torch.randn(n, gan.dim_latente, 1, 1, device=dispositivo)
        falsas.extend(tensor_to_numpy(t) for t in gan.generar(z).cpu())
    return calcular_metricas_calidad(reales_metricas, falsas, "cuda", feats_reales=feats_reales)

def a_float(v):
    try: return float(str(v).replace("%", ""))
    except: return np.nan

def mostrar(epoca, seg_por_epoca):
    clear_output(wait=True)
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    grid = vutils.make_grid(gan.generar(gan.ruido_fijo[:9]).cpu(), nrow=3, normalize=True, value_range=(-1, 1))
    ax[0].imshow(grid.permute(1, 2, 0).numpy()); ax[0].axis("off")
    ax[0].set_title(f"Época {epoca}/{EPOCAS} · {gan.pasos_totales:,} it · {seg_por_epoca:.1f}s/época")

    ax[1].plot(historial["epoca"], historial["loss_G"], label="Pérdida G")
    ax[1].plot(historial["epoca"], historial["loss_D"], label="Pérdida D")
    ax[1].set_title("Pérdidas"); ax[1].set_xlabel("Época"); ax[1].legend(); ax[1].grid(alpha=.3)

    if historial["ep_met"]:
        ax[2].plot(historial["ep_met"], historial["kid"], "o-", color="#8B5CF6", label="KID×1000 (↓)")
        ax[2].set_ylabel("KID×1000")
        ax2b = ax[2].twinx()
        ax2b.plot(historial["ep_met"], historial["div"], "s--", color="#10B981", label="Diversidad % (↑)")
        ax2b.set_ylim(0, max(110, np.nanmax(historial["div"]) * 1.1)); ax2b.set_ylabel("Diversidad %")
        lineas = ax[2].get_lines() + ax2b.get_lines()
        ax[2].legend(lineas, [l.get_label() for l in lineas], loc="upper right")
    ax[2].set_title("Calidad"); ax[2].set_xlabel("Época"); ax[2].grid(alpha=.3)
    plt.tight_layout(); plt.show()

    if historial["ep_met"]:
        print(f"Última métrica (época {historial['ep_met'][-1]}): KID×1000={historial['kid'][-1]:.2f} | "
              f"FID={historial['fid'][-1]:.1f} | Diversidad={historial['div'][-1]:.0f}%")
    restante = (EPOCAS - epoca) * seg_por_epoca / 60
    print(f"Tiempo restante estimado: {restante:.1f} min")

def guardar(epoca):
    gan.guardar_checkpoint(epoca, ruta_dir=DIR_CHECKPOINTS)
    torch.save(historial, os.path.join(DIR_CHECKPOINTS, "historial.pt"))
    # Conservar solo los 3 checkpoints más recientes para no llenar Drive
    cks = sorted(glob.glob(os.path.join(DIR_CHECKPOINTS, "fastgan_epoca_*.pth")),
                 key=lambda p: int(p.split("_")[-1].split(".")[0]))
    for viejo in cks[:-3]:
        os.remove(viejo)

t0 = time.time()
try:
    for epoca in range(epoca_inicio, EPOCAS):
        t_ep = time.time()
        loss_G, loss_D = gan.entrenar_epoca(cargador, epoca, EPOCAS)
        num_ep = epoca + 1
        historial["epoca"].append(num_ep); historial["loss_G"].append(loss_G); historial["loss_D"].append(loss_D)

        if num_ep == 1 or num_ep % INTERVALO_METRICAS == 0 or num_ep == EPOCAS:
            m = calcular_metricas()
            historial["ep_met"].append(num_ep)
            historial["kid"].append(a_float(m.get("kid")))
            historial["fid"].append(a_float(m.get("fid")))
            historial["div"].append(a_float(m.get("diversidad")))

        if num_ep % INTERVALO_CHECKPOINT == 0 or num_ep == EPOCAS:
            guardar(num_ep)

        if num_ep == 1 or num_ep % INTERVALO_VISTA_PREVIA == 0 or num_ep == EPOCAS:
            mostrar(num_ep, time.time() - t_ep)
except KeyboardInterrupt:
    print("⏹ Interrumpido: guardando checkpoint...")
    guardar(historial["epoca"][-1] if historial["epoca"] else 0)

print(f"✔ Listo en {(time.time() - t0) / 60:.1f} min. Checkpoints en: {DIR_CHECKPOINTS}")

## 8. Vista previa grande (generador EMA)

In [ ]:
z = torch.randn(64, gan.dim_latente, 1, 1, device=dispositivo)
grid = vutils.make_grid(gan.generar(z).cpu(), nrow=8, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(12, 12)); plt.imshow(grid.permute(1, 2, 0).numpy()); plt.axis("off")
plt.title(f"64 muestras sintéticas · clase '{CLASE}'"); plt.show()

## 9. Generar imágenes sintéticas y descargar
Usa `generar_lote_sintetico` de tu código (generador EMA). Se descarga un ZIP con las imágenes y el checkpoint final,
que puedes cargar en la app de escritorio con `EntrenadorFastGAN.cargar_checkpoint(...)` (funciona en CPU).

In [ ]:
from google.colab import files

dir_sinteticas = os.path.join(DIR_SALIDA, "synthetic")
rutas = gan.generar_lote_sintetico(CANTIDAD_SINTETICAS, dir_sinteticas, clase_nombre=CLASE, batch_size=128)

ck_final = gan.guardar_checkpoint(historial["epoca"][-1] if historial["epoca"] else 0, ruta_dir=DIR_CHECKPOINTS)
ruta_zip = shutil.make_archive(f"/content/gania_{CLASE}_sinteticas", "zip", os.path.join(dir_sinteticas, CLASE))
print(f"{len(rutas)} imágenes → {ruta_zip}")
print("Checkpoint final →", ck_final)

files.download(ruta_zip)
files.download(ck_final)